In [1]:
# DEPENDENCIES

import sys
sys.path.insert(0, '..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

In [2]:
# SET-UP

# Exctract the subject names
ALICE_SUBJECTS = helper_functions.alice_get_subjects()
FUGLSANG_SUBJECTS = helper_functions.fuglsang_get_subjects()

In [3]:
# CREATE SET OF ALICE TRFS

# Load backward TRFs for all subjects
trf_types = [
    # Single predictors
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD, False),

    # Multiple predictors
    #([PREDICTOR_TYPE.ENVELOPE, PREDICTOR_TYPE.ENVELOPE_ONSET], ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD, False),
]

trf_data, n_subjects = helper_functions.load_trfs(DATASET_TYPE.ALICE, ALICE_SUBJECTS, trf_types, trf_dir=ALICE_TRF_DIR)
#testing_trf_data, n_testing_subjects = helper_functions.load_trfs(testing_subjects, trf_types, trf_dir=FUGLSANG_TRF_DIR)

print('\nProcess Completed! \n:)')

  ✓ S01
  ✓ S03
  ✓ S04
  ✓ S05
  ✓ S06
  ✓ S08
  ✓ S10
  ✓ S11
  ✓ S12
  ✓ S13
  ✓ S14
  ✓ S15
  ✓ S16
  ✓ S17
  ✓ S18
  ✓ S19
  ✓ S20
  ✓ S21
  ✓ S22
  ✓ S25
  ✓ S26
  ✓ S34
  ✓ S35
  ✓ S36
  ✓ S37
  ✓ S38
  ✓ S39
  ✓ S40
  ✓ S41
  ✓ S42
  ✓ S44
  ✓ S45
  ✓ S48

Loaded: 33 subjects | Skipped: 0 subjects

Process Completed! 
:)


In [4]:
# FIND THE THREE CHANNELS TO REMOVE FROM FUGLSANG ELECTRODE LAYOUT

# Choose an arbitrary subject
subject = 'S13'

# ── 1. Load the data ──────────────────────────────────────────────────────────
# Alice is a .fif file (MNE format), Fuglsang is an eelbrain pickle
alice_raw = mne.io.read_raw_fif(ALICE_EEG_DIR / subject / f'{subject}_alice-raw.fif', preload=True)
fuglsang_raw = eelbrain.load.unpickle(FUGLSANG_EEG_DIR / subject / f'{subject}_eeg.pickle')

# ── 2. Get channel names ───────────────────────────────────────────────────────
# Alice channels are named '1'-'61' (numeric), plus VEOG and AUD (non-EEG) → keep EEG only
# Fuglsang channels are already named with standard 10-10 labels (e.g. 'Cz', 'Fp1', ...)
alice_eeg_picks = mne.pick_types(alice_raw.info, eeg=True)
alice_ch_names = [alice_raw.ch_names[i] for i in alice_eeg_picks]  # ['1','2',...,'61']
fuglsang_ch_names = fuglsang_raw.sensor.names                       # ['Fp1','AF7',...]

print(f"Alice:    {len(alice_ch_names)} EEG channels: {alice_ch_names[:5]}...")
print(f"Fuglsang: {len(fuglsang_ch_names)} channels:     {fuglsang_ch_names[:5]}...")

# ── 3. Load Alice's physical electrode positions ───────────────────────────────
# The .sfp file is the custom EasyCap M10 montage used when Alice was recorded.
# It contains the 3D (x,y,z) position of each electrode on the scalp.
# The channel names in this file are also numeric ('1'-'61'), matching alice_raw.
montage = mne.channels.read_custom_montage(
    'easycapM10-acti61_elec.sfp'  # adjust path if needed
)
alice_positions = montage.get_positions()['ch_pos']  # dict: '1' -> (x,y,z), etc.

# ── 4. Load BioSemi64 standard positions ──────────────────────────────────────
# BioSemi64 is the cap used by Fuglsang. MNE has this built-in as a standard montage.
# Its positions are in metres; Alice's .sfp is in centimetres scaled to a unit sphere.
# We'll use Cz (channel '33' in Alice, known to be at the very top of the head)
# to calculate the scale factor between the two coordinate systems.
biosemi64 = mne.channels.make_standard_montage('biosemi64')
biosemi_positions = biosemi64.get_positions()['ch_pos']  # dict: 'Cz' -> (x,y,z), etc.

cz_alice   = np.linalg.norm(alice_positions['33'])       # Alice '33' = Cz
cz_biosemi = np.linalg.norm(biosemi_positions['Cz'])
scale = cz_alice / cz_biosemi                            # ~94.7x
print(f"\nCoordinate scale factor (Alice sfp / BioSemi metres): {scale:.1f}")

# ── 5. Build a cost matrix of distances between all pairs ─────────────────────
# Rows = Alice's 61 channels, Cols = BioSemi64's 64 channels
# Each cell = Euclidean distance between that Alice electrode and that BioSemi electrode
# (BioSemi positions are multiplied by the scale factor so units match)
alice_nums    = sorted(alice_positions.keys(), key=int)   # ['1','2',...,'61']
biosemi_names = list(biosemi_positions.keys())             # ['Fp1','AF7',...]

cost_matrix = np.zeros((len(alice_nums), len(biosemi_names)))
for i, num in enumerate(alice_nums):
    for j, name in enumerate(biosemi_names):
        alice_xyz   = np.array(alice_positions[num])
        biosemi_xyz = np.array(biosemi_positions[name]) * scale
        cost_matrix[i, j] = np.linalg.norm(alice_xyz - biosemi_xyz)

# ── 6. Solve the optimal one-to-one matching (Hungarian algorithm) ────────────
# We want to assign each Alice channel to exactly one BioSemi64 channel,
# minimising the total distance across all 61 pairs simultaneously.
# This avoids two Alice channels both claiming the same BioSemi name.
row_ind, col_ind = linear_sum_assignment(cost_matrix)

# Build the final mapping: Alice numeric name -> best BioSemi 10-10 name
numeric_to_10_10 = {}
for i, j in zip(row_ind, col_ind):
    alice_num   = alice_nums[i]
    biosemi_name = biosemi_names[j]
    dist         = cost_matrix[i, j]
    numeric_to_10_10[alice_num] = (biosemi_name, dist)

print("\nAlice channel mapping (numeric -> 10-10 name):")
for num in sorted(numeric_to_10_10, key=int):
    name, dist = numeric_to_10_10[num]
    print(f"  Alice '{num:>2}' -> '{name:<6}'  (distance: {dist:.3} mm)")

# ── 7. Find the 3 unmatched BioSemi64 channels ────────────────────────────────
# 61 Alice channels can only claim 61 of the 64 BioSemi64 slots.
# The 3 leftover BioSemi64 channels have no spatial equivalent in Alice → drop them.
matched_biosemi = {v[0] for v in numeric_to_10_10.values()}
channels_to_drop = sorted(set(biosemi_names) - matched_biosemi)

print(f"\nBioSemi64 channels with no Alice equivalent (drop from Fuglsang):")
print(f"   {channels_to_drop}")

Alice:    56 EEG channels: ['1', '2', '3', '4', '5']...
Fuglsang: 64 channels:     ['Fp1', 'AF7', 'AF3', 'F1', 'F3']...

Coordinate scale factor (Alice sfp / BioSemi metres): 1.0

Alice channel mapping (numeric -> 10-10 name):
  Alice ' 1' -> 'FCz   '  (distance: 6.54e-06 mm)
  Alice ' 2' -> 'FC2   '  (distance: 0.0187 mm)
  Alice ' 3' -> 'CP2   '  (distance: 0.0187 mm)
  Alice ' 4' -> 'CPz   '  (distance: 6.54e-06 mm)
  Alice ' 5' -> 'CP1   '  (distance: 0.0187 mm)
  Alice ' 6' -> 'FC1   '  (distance: 0.0187 mm)
  Alice ' 7' -> 'AFz   '  (distance: 0.00165 mm)
  Alice ' 8' -> 'AF4   '  (distance: 0.0101 mm)
  Alice ' 9' -> 'F4    '  (distance: 0.0188 mm)
  Alice '10' -> 'FC6   '  (distance: 0.0081 mm)
  Alice '11' -> 'C6    '  (distance: 0.0094 mm)
  Alice '12' -> 'CP6   '  (distance: 0.0155 mm)
  Alice '13' -> 'P4    '  (distance: 0.014 mm)
  Alice '14' -> 'POz   '  (distance: 0.0186 mm)
  Alice '15' -> 'PO3   '  (distance: 0.0226 mm)
  Alice '16' -> 'P3    '  (distance: 0.014 mm)
  

In [5]:
# REMOVE CHANNELS C1, C2, P04 FROM ALL FUGLSANG EEG DATA

# Define channels to remove as a list
channels_to_drop = ['C1', 'C2', 'PO4']

for subject in FUGLSANG_SUBJECTS:
    eeg = eelbrain.load.unpickle(FUGLSANG_EEG_DIR / subject / f'{subject}_eeg.pickle')

    # Confirm the channels are actually present before trying to drop
    available = set(eeg.sensor.names)
    to_drop_here = [ch for ch in channels_to_drop if ch in available]
    missing = [ch for ch in channels_to_drop if ch not in available]
    
    if missing:
        print(f"  [{subject}] WARNING Channels not found (already removed?): {missing}")

    # We select all sensors EXCEPT the ones we want to drop
    sensors_to_keep = [ch for ch in eeg.sensor.names if ch not in channels_to_drop]
    eeg_trimmed = eeg.sub(sensor=sensors_to_keep)
    print(f"  [{subject}] {len(eeg.sensor.names)} → {len(eeg_trimmed.sensor.names)} channels ✓")

    # ————————————————————————————————————————————————————————————————————————————————————————————————
    # Save pickles
    eelbrain.save.pickle(eeg_trimmed, FUGLSANG_EEG_DIR / subject / f'{subject} eeg_trimmed.pickle')

print('\nProcess Completed! \n:)')


  [S1] 64 → 61 channels ✓
  [S2] 64 → 61 channels ✓
  [S3] 64 → 61 channels ✓
  [S4] 64 → 61 channels ✓
  [S5] 64 → 61 channels ✓
  [S6] 64 → 61 channels ✓
  [S7] 64 → 61 channels ✓
  [S8] 64 → 61 channels ✓
  [S9] 64 → 61 channels ✓
  [S10] 64 → 61 channels ✓
  [S11] 64 → 61 channels ✓
  [S12] 64 → 61 channels ✓
  [S13] 64 → 61 channels ✓
  [S14] 64 → 61 channels ✓
  [S15] 64 → 61 channels ✓
  [S16] 64 → 61 channels ✓
  [S17] 64 → 61 channels ✓
  [S18] 64 → 61 channels ✓

Process Completed! 
:)


In [6]:
# MAKE UNIVERSAL ALICE TRFS WITH RENAMED CHANNELS TO MATCH FUGLSANG

# Alice TRFs have numeric channel names ('1'-'61') in Alice's order.
# Fuglsang EEG has 10-10 names in Fuglsang's order.
# numeric_to_10_10 maps Alice numeric -> 10-10 name (computed earlier via Hungarian matching).
# For convolution to be correct, TRF row i must correspond to EEG channel i.
# So we reorder the TRF rows to match Fuglsang's channel order.

# Define arbitrary subject
subject = 'S13'

# Extract model names
models = []
for trf_type in trf_types:
    models.append(helper_functions.get_trf_model_name(DATASET_TYPE.ALICE, *trf_type))

# Load one Fuglsang EEG as reference for the target channel order
ref_eeg = eelbrain.load.unpickle(FUGLSANG_EEG_DIR / subject / f'{subject} eeg_trimmed.pickle')
name_to_numeric = {name: num for num, (name, dist) in numeric_to_10_10.items()}

# -----------------------------------------------------------------------------------------------------
# Loop over all models
for model in models:
    print(f'\nProcessing universal TRF for model: {model}')

    # Collect TRFs for all subjects for this model
    trf_list = [trf.h_scaled for trf in trf_data[model]]
    print(trf_list)

    # Combine and average across subjects
    universal_trf = eelbrain.combine(trf_list).mean('case')

    # Reorder channels to match Fuglsang's channel order and assign Fuglsang's sensor dimension
    # so the universal TRF can be convolved with Fuglsang EEG directly
    trf_channel_order = [universal_trf.sensor.names.index(name_to_numeric[ch]) for ch in ref_eeg.sensor.names]
    universal_trf_aligned = eelbrain.NDVar(
        universal_trf.x[trf_channel_order, :],
        (ref_eeg.sensor, universal_trf.time),
        name=universal_trf.name
    )

    # Save universal TRF for this model
    eelbrain.save.pickle(universal_trf_aligned, ALICE_TRF_DIR / f'64hz-universal-{model}.pickle')

    print(f'Saved universal TRF for model {model}')


Processing universal TRF for model: decoder-envelope_log
[<NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 sensor, 73 time>, <NDVar: 61 se

In [7]:
# MAKE PREDICTED ENVELOPES USING ALICE TRFs ON FUGLSANG DATA AND COMPUTE CORRELATION

# Define which Fuglsang models to run
trf_types = [
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, False),
]

models = [helper_functions.get_trf_model_name(DATASET_TYPE.FUGLSANG, *t) for t in trf_types]

# Run across all subjects and models ─────────────────────────────────────
r_values  = {model: [] for model in models}
r2_values = {model: [] for model in models}

for subject in FUGLSANG_SUBJECTS:
    # Load this subject's EEG (61 channels, Fuglsang order)
    eeg = eelbrain.load.unpickle(FUGLSANG_EEG_DIR / subject / f'{subject} eeg_trimmed.pickle')

    for model in models:
        # Resolve predictor type from model name
        alice_predictor_name = model.split('_')[2]
        predictor_type = (PREDICTOR_TYPE.ENVELOPE if alice_predictor_name == PREDICTOR_TYPE.ENVELOPE.value
                          else PREDICTOR_TYPE.ENVELOPE_ONSET)

        # Load the Alice universal TRF (trained on all Alice subjects, numeric channel names)
        alice_model_name = helper_functions.get_trf_model_name(
            DATASET_TYPE.ALICE, predictor_type, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD)
        trf = eelbrain.load.unpickle(ALICE_TRF_DIR / f'64hz-universal-{alice_model_name}.pickle')

        # Load the Fuglsang predictor (attended or ignored envelope)
        fuglsang_predictor_name = model.split('backward_')[1]
        predictor = eelbrain.load.unpickle(
            FUGLSANG_PREDICTOR_DIR / subject / f'{fuglsang_predictor_name}_concat.pickle')

        # Convolve TRF with EEG -> predicted envelope (1D time series)
        predicted_envelope = eelbrain.convolve(trf, eeg)

        # Correlate predicted envelope with actual envelope
        r  = np.corrcoef(predictor.x, predicted_envelope.x)[0, 1]
        r2 = r ** 2

        r_values[model].append(r)
        r2_values[model].append(r2)

# Summary ────────────────────────────────────────────────────────────────
print('\n===== Universal decoder summary =====')
for model in models:
    mean_r  = np.mean(r_values[model])
    std_r   = np.std(r_values[model])
    mean_r2 = np.mean(r2_values[model])
    std_r2  = np.std(r2_values[model])
    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f},  r² = {mean_r2:.4f} ± {std_r2:.4f}')


===== Universal decoder summary =====
backward_attended_envelope: r = 0.0514 ± 0.0415,  r² = 0.0044 ± 0.0047
backward_ignored_envelope: r = 0.0285 ± 0.0219,  r² = 0.0013 ± 0.0012
backward_attended_envelope_onset: r = 0.0171 ± 0.0112,  r² = 0.0004 ± 0.0004
backward_ignored_envelope_onset: r = 0.0114 ± 0.0070,  r² = 0.0002 ± 0.0002


In [10]:
# CLASSIFICATION

def aad_classifier_universal(predictor, attention_att, attention_ign):

    att_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor, attention_att, MODEL_TYPE.BACKWARD, False
    )
    ign_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor, attention_ign, MODEL_TYPE.BACKWARD, False
    )

    # Load universal TRF — one per predictor type, trained on Alice data
    alice_model_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.ALICE, predictor, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD
    )
    trf = eelbrain.load.unpickle(ALICE_TRF_DIR / f'64hz-universal-{alice_model_name}.pickle')

    correct = 0
    total   = 0

    for subject in FUGLSANG_SUBJECTS:

        # ----------------------------------------
        # Load EEG
        # ----------------------------------------
        eeg = eelbrain.load.unpickle(
            FUGLSANG_EEG_DIR / subject / f'{subject} eeg_trimmed.pickle'
        )

        # ----------------------------------------
        # Load true stimuli
        # ----------------------------------------
        att_stim = eelbrain.load.unpickle(
            FUGLSANG_PREDICTOR_DIR / subject / f'{att_name.split("backward_")[1]}_concat.pickle'
        )

        ign_stim = eelbrain.load.unpickle(
            FUGLSANG_PREDICTOR_DIR / subject / f'{ign_name.split("backward_")[1]}_concat.pickle'
        )

        # Reconstruct envelope once (single universal TRF)
        pred = eelbrain.convolve(trf, eeg).x

        # Correlate against attended and ignored stimuli
        r_att = np.corrcoef(pred, att_stim.x)[0, 1]
        r_ign = np.corrcoef(pred, ign_stim.x)[0, 1]

        # Decision: does the predicted envelope match the attended stream better?
        correct += (r_att > r_ign)
        total += 1

        print(f"{subject}: r_att={r_att:.3f}, r_att_ign={r_ign:.3f}")

    acc = correct / total if total > 0 else 0

    print(f"\n✅ Classification rate ({att_name} vs {ign_name}): {acc:.2%}")
    print('\n————————————————————————————————————————————————————————-\n')

    return acc


# Envelope backward model
aad_classifier_universal(PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.ATTENDED, ATTENTION_TYPE.IGNORED)

# Envelope onset backward model
aad_classifier_universal(PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, ATTENTION_TYPE.IGNORED)

S1: r_att=0.038, r_att_ign=0.020
S2: r_att=0.053, r_att_ign=0.020
S3: r_att=0.049, r_att_ign=0.018
S4: r_att=0.095, r_att_ign=0.060
S5: r_att=0.024, r_att_ign=0.016
S6: r_att=0.037, r_att_ign=0.019
S7: r_att=0.128, r_att_ign=0.046
S8: r_att=0.085, r_att_ign=0.039
S9: r_att=0.035, r_att_ign=0.016
S10: r_att=0.020, r_att_ign=0.018
S11: r_att=-0.007, r_att_ign=0.000
S12: r_att=0.034, r_att_ign=0.027
S13: r_att=0.072, r_att_ign=0.053
S14: r_att=0.078, r_att_ign=0.041
S15: r_att=0.100, r_att_ign=0.050
S16: r_att=0.006, r_att_ign=0.060
S17: r_att=-0.032, r_att_ign=-0.029
S18: r_att=0.111, r_att_ign=0.038

✅ Classification rate (backward_attended_envelope vs backward_ignored_envelope): 83.33%

————————————————————————————————————————————————————————-

S1: r_att=0.019, r_att_ign=0.015
S2: r_att=0.024, r_att_ign=0.004
S3: r_att=0.008, r_att_ign=0.002
S4: r_att=0.035, r_att_ign=0.023
S5: r_att=0.005, r_att_ign=0.002
S6: r_att=0.010, r_att_ign=-0.000
S7: r_att=0.053, r_att_ign=0.024
S8: r_att=0.0

np.float64(0.8333333333333334)